In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

In [ ]:
def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res

In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('merged_corrected.xlsx')
paths = list(df['path_name'].value_counts().index.values)
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
    df.loc[df['path_name']==path, "path_ma_O"] = df.loc[df['path_name']==path, 'd18o'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma']] = df_sorted[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).std()


df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_absolute'])
df_resampled_ = df_sorted.set_index('age_dt')
df_resampled_.drop(['Name', 'Mass', 'Amp 44', 'Area 44', 'path_name',
       'path_num', 'path_ma','path_ma_O', 'age_ws', 'age_ws2',
       'd13c_ma', 'd18o_ma', 'd13c_ma2', 'd18o_ma2', 'd13c_std', 'd18o_std'], axis=1, inplace=True)
df_resampled = df_resampled_.resample('30D').mean()

df_resampled.interpolate(inplace=True)
df_resampled[['d13c_ma','d18o_ma']] = df_resampled[['d13c','d18o']].rolling(window_size, center=True).mean()
df_resampled[['d13c_ma2','d18o_ma2']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).std()

df_resampled

In [ ]:
fred = pd.read_excel('../2025_08_22_FredsData-post2008/FredData_editedMemo.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['d13c_ma','d18o_ma']] = fred[['d13c','d18o_pdb']].rolling(window_size, center=True).mean()
fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[3:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['decimal_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred_resampled

In [ ]:
enso_phase = pd.read_csv('../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')
enso_phase

In [ ]:
enso = pd.read_csv("../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso

In [ ]:
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
enso

In [ ]:
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)
blocks

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')


#for i, index in enumerate(blocks.index):
    
    #df_iter = enso_phase.loc[index]
    #df_iter_next = enso_phase.loc[index+1]
    #fig.add_vrect(x0=df_iter['Year'], x1=df_iter_next['Year'], fillcolor=phase_color[df_iter['Phase']], opacity=0.1)
for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)





# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
fig.write_html('fred_d13C_ENSO.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers",
                         marker_color="lightblue",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'], mode="lines",
                         line_color='blue', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')



for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)



# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18O and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta O ^{178}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
#fig.write_html('d18o_ENSO.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue'}
enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]
paths = list(df['path_name'].value_counts().index.values)
colors_path_dict = dict(zip(paths , cl.n_colors('rgb(0,0,255)', 'rgb(255,0,0)', len(paths), colortype='rgb')))
colors_path = [colors_path_dict[cat] for cat in df['path_name']]


#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df['age_absolute'], y=df['d13c'],  mode="lines+markers", marker=dict(color=colors_path), text=df['Name'], name='d13C'),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


fig.add_vline(x=1935.25, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1935 Mar')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')


for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]
    fig.add_vrect(x0=df_iter['Year'], x1=df_iter_next['Year'], fillcolor=phase_color[df_iter['Phase']], opacity=0.1)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()


In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]

In [ ]:
fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df[df['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_absolute'], y=df_iter['d18o'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    fig.add_trace(go.Scatter(x=df_iter['age_absolute'], y=df_iter['path_ma_O'], mode='lines',marker=dict(color=colors[i]), showlegend=False))
fig.show()
#fig.write_html('d13C_paths.html')

In [ ]:
fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df[df['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_absolute'], y=df_iter['d13c'], mode='markers', marker=dict(color=colors[i]), name=path))
    fig.add_trace(go.Scatter(x=df_iter['age_absolute'], y=df_iter['path_ma'], mode='lines',marker=dict(color=colors[i]), showlegend=False))
fig.show()
#fig.write_html('d13C_paths.html')

In [ ]:
df[df['path_name']==path]

In [ ]:
for i, path in enumerate(paths):
    df_iter = df[df['path_name']==path]
    res = df_iter[['d13c','d18o']].describe()
    print(path+' RESULTS'+'\n')
    print(res)
    print('\n')

In [ ]:
df_iter.describe()

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers",
                         marker_color="lightblue",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'], mode="lines",
                         line_color='blue', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d18o_ma2'], mode="lines",
                         line_color='orange', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.66], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE


fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.6], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1992.5, y=25.7, text="?", showarrow=False, font=dict(size=35, color="red"))

fig.add_annotation(x=1991.4, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=20, ay=-50)

fig.add_annotation(x=1974.3, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-20, ay=40)


#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)



# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18O and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{178}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
#fig.write_html('d18o_ENSO.html') 


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
#fig.write_html('fred_d13C_ENSO.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
2007+(31+29+31)/365

## RESAMPLED

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o'],  mode="markers", marker_color="lightblue",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_std'], mode="lines", line_color='darkred', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_std'], mode="lines", line_color='darkred', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.66], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE


fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.6], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1992.5, y=25.7, text="?", showarrow=False, font=dict(size=35, color="red"))

fig.add_annotation(x=1991.4, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=20, ay=-50)

fig.add_annotation(x=1974.3, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-20, ay=40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta O ^{18}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d18o_std.html')


In [ ]:
fig, axes = plt.subplots(2, figsize=(10,6), sharex=True,height_ratios=[2,1], gridspec_kw={'hspace':0})

ax = axes[0]

ax.plot(df_resampled['age_absolute'], df_resampled['d13c'], ".", color='#6EEB6A', alpha=0.7, label=r'$\delta^{13}C$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma'],  color='#368033', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma2'],  color='red', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d13c'], ".", color='#D6F6C0', alpha=1, label=r'fred $\delta^{13}C$')
ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#459014', lw=0.75 )


ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-5, ymax=5, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(-3, 0.5)

ax.grid()
ax.legend()
ax.set_ylabel('')


ax = axes[1]

ax.plot(df_resampled['age_absolute'], df_resampled['d13c_std'], color='darkred' , label=r'std $\delta^{13}C$'  )
ax.plot(fred['decimal_year'], fred['d13c_std'], '--', color='darkred' , label= r'fred std $\delta^{13}C$')

ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(0.1,0.8)
ax.grid()
ax.legend()



In [ ]:
fig, axes = plt.subplots(2, figsize=(10,6), sharex=True,height_ratios=[2,1], gridspec_kw={'hspace':0})

ax = axes[0]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}O$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(25, 26.57)

ax.grid()
ax.legend()



ax = axes[1]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o_std'], color='darkred' , label='std d18O'  )
ax.plot(fred['decimal_year'], fred['d18o_std'], '--', color='darkred' , label='fred std d18O'  )

ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(-0.05,0.35)
ax.grid()
ax.legend()





In [ ]:
window = 10
col = 'd18o'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])

window = 5
col = 'd13c'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])



In [ ]:
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
statsmodels

In [ ]:
# Store centered coefficients
coefs_centered = pd.DataFrame()

# Define multiple windows and target columns
windows = [36, 36]
cols = ['d18o', 'd13c']

for window, col in zip(windows, cols):
    
    # Prepare data (assumes df_resampled contains all variables)
    data = df_resampled[[col, 'age_absolute']].dropna()

    if len(data) < window:
        print(f"Not enough data for column '{col}' with window {window}. Skipping.")
        continue
    
    # Setup endogenous and exogenous variables
    endog = data[col]
    exog = sm.add_constant(data['age_absolute'])

    # Fit rolling OLS
    rols = RollingOLS(endog, exog, window=window)
    rres = rols.fit()

    # Shift result to center and rename columns
    coefs = rres.params.shift(-(window // 2)).dropna()
    coefs = coefs.rename(columns={'const': f'{col}_intercept', 'age_absolute': f'{col}_slope'})

    # Merge into result
    coefs_centered = coefs_centered.join(coefs, how='outer')

# Add shared age column (assumes time index is same)
coefs_centered['age_absolute'] = df_resampled['age_absolute']

# Drop rows with NaNs in all slope columns (optional)
slope_cols = [f"{col}_slope" for col in cols]
coefs_centered.dropna(subset=slope_cols, how='all', inplace=True)

coefs_centered[f'd13c_slope'].plot(title='Centered Rolling Slope (d13c)')
plt.axhline(0, linestyle='--', color='gray')
plt.xlabel('Time Index')
plt.ylabel('Slope')
plt.grid(True)
plt.show()

In [ ]:
coefs_centered

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)


fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)

#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
fig, axes = plt.subplots(3, figsize=(10,6), sharex=True,height_ratios=[3,1,1], gridspec_kw={'hspace':0})

ax = axes[0]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}O$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(25, 26.57)

ax.grid()
ax.legend(loc='lower left')



ax = axes[1]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o_std'], color='darkred' , label='std d18O'  )
ax.plot(fred['decimal_year'], fred['d18o_std'], '--', color='darkred' , label='fred std d18O'  )

ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(-0.05,0.35)
ax.grid()
ax.legend(loc='lower left')



ax = axes[2]

ax.plot(coefs_centered['age_absolute'], coefs_centered['d13c_slope'], color='pink' , label='moving OLS'  )

ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(-0.5,0.5)
ax.grid()
ax.legend(loc='lower left')


In [ ]:
def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

In [ ]:
d13c_fft_freq, d13c_fft = FFT_res(df_resampled['d13c'], 30)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d18o'], 30)
enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

ax.plot(1/d13c_fft_freq[1:], fft_noramlize(d13c_fft[1:])**2,label='d13C')
ax.plot(1/d18o_fft_freq[1:], fft_noramlize(d18o_fft[1:])**2,label='d18O')

ax.vlines(365, ymin=-100, ymax=2500, color='red', alpha=0.4, label='1 Y')

#ax.set_xlim(200,2000)
ax.set_xlim(200,3600)
#ax.set_xscale('log')
ax.set_ylim(0,0.001)
ax.legend()

In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_fft_freq[1:], fft_noramlize(d18o_fft[1:])**2,label='d18O')
    if i==1:
        ax.plot(1/d13c_fft_freq[1:], fft_noramlize(d13c_fft[1:])**2, color='green',label='d13C')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:])**3, color='gray', label='enso')
    ax.legend(loc='center left')  
    #days_freq = 271
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    #days_freq = 281
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')



    ax.set_xlim(200,400)
#ax.set_xlim(200,3600)
#ax.set_xscale('log')
    ax.set_ylim(0,0.003)
    plt.yticks([0,0.001, 0.002])
axes[0].set_ylim(0,0.000001)
#plt.yticks([0.5e-6, 1e-6])
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum')

#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')



In [ ]:
df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window=window_size2, center=True).std()
d13c_stds = df_resampled.dropna()['d13c_std']
d18o_stds = df_resampled.dropna()['d18o_std']
d13c_std_fft_freq, d13c_std_fft = FFT_res(d13c_stds, 30)
d18o_std_fft_freq, d18o_std_fft = FFT_res(d18o_stds, 30)


In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

ax.plot(1/d18o_std_fft_freq[1:], fft_noramlize(d18o_std_fft[1:])**2,label='d18O')
ax.plot(1/d13c_std_fft_freq[1:], fft_noramlize(d13c_std_fft[1:])**2, color='green',label='d13C')
ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:])**3, color='gray', label='enso')



days_freq = 271
ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
days_freq = 281
ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')



ax.set_xlim(200,400)
#ax.set_xlim(200,3600)
#ax.set_xscale('log')
ax.set_ylim(0,0.003)
ax.set_xlabel('Period [days]')
ax.set_ylabel('Normalized FFT Amplitude')
ax.legend()
ax.set_title('Moving Standard Deviation FFT')

#fig.savefig('std_fft.pdf')


In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_std_fft_freq[1:], fft_noramlize(d18o_std_fft[1:])**2,label=r'$\delta^{18}O$')
    if i==1:
        ax.plot(1/d13c_std_fft_freq[1:], fft_noramlize(d13c_std_fft[1:])**2, color='green',label=r'$\delta^{13}C$')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:])**3, color='gray', label='enso')
    
    ax.legend(loc='upper right')  
    days_freq = 271
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 281
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')

    days_freq = 746    
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    
    
    ax.set_xlim(500,4000)
    #ax.set_xlim(200,3600)
    #ax.set_xscale('log')
    plt.yticks([0,0.001])
    ax.set_ylim(0,0.004)

axes[2].set_ylim(0,1)
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum ')
#ax.legend(loc=[500,0])
#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')


In [ ]:
1700/365

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))

ax.plot(1/d13c_std_fft_freq[1:], fft_noramlize(d13c_std_fft[1:])**2,label='d13C')
ax.plot(1/d18o_std_fft_freq[1:], fft_noramlize(d18o_std_fft[1:])**2,label='d18O')
ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:])**3,label='enso')



days_freq = 271
ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
days_freq = 281
ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')



ax.set_xlim(500, 900)
#ax.set_xlim(200,3600)
#ax.set_xscale('log')
ax.set_ylim(0,0.005)
ax.set_xlabel('Period [days]')
ax.set_ylabel('Normalized FFT Amplitude')
ax.legend()
ax.set_title('Moving Standard Deviation FFT')

#fig.savefig('std_fft.pdf')
